In [ ]:
from deap import creator, base, tools
import random

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
# Attribute generator
toolbox.register("attr_bool", random.randint, 0, 1)
# Structure initializers
toolbox.register("individual", tools.initRepeat, creator.Individual,
    toolbox.attr_bool, 100)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evalOneMax(individual):
    return sum(individual),

toolbox.register("evaluate", evalOneMax)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

pop = toolbox.population(n=300)
fitnesses = list(map(toolbox.evaluate, pop))

print(fitnesses)

for ind, fit in zip(pop, fitnesses):
    ind.fitness.values = fit

CXPB, MUTPB = 0.5, 0.2
fits = [ind.fitness.values[0] for ind in pop]

# Variable keeping track of the number of generations
g = 0

# Begin the evolution
while max(fits) < 100 and g < 1000:
    # A new generation
    g = g + 1
    print("-- Generation %i --" % g)
    # Select the next generation individuals
    offspring = toolbox.select(pop, len(pop))
    # Clone the selected individuals
    offspring = list(map(toolbox.clone, offspring))
    # Apply crossover and mutation on the offspring
    for child1, child2 in zip(offspring[::2], offspring[1::2]):
        if random.random() < CXPB:
            toolbox.mate(child1, child2)
            del child1.fitness.values
            del child2.fitness.values

    for mutant in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(mutant)
            del mutant.fitness.values

    # Evaluate the individuals with an invalid fitness
    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = map(toolbox.evaluate, invalid_ind)
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    pop[:] = offspring

    # Gather all the fitnesses in one list and print the stats
    fits = [ind.fitness.values[0] for ind in pop]

    length = len(pop)
    mean = sum(fits) / length
    sum2 = sum(x*x for x in fits)
    std = abs(sum2 / length - mean**2)**0.5

    print("  Min %s" % min(fits))
    print("  Max %s" % max(fits))
    print("  Avg %s" % mean)
    print("  Std %s" % std)

In [4]:
import numpy as np
from deap import base, creator, tools
import random
objectives = [8,3,4,1,4,76]
fitnesses = [1,6,23,1,2,6]

creator.create("FitnessMin", base.Fitness, weights=(-1.0, -1.0, -1.0))
creator.create("Individual", list, fitness=creator.FitnessMin)
ind1 = creator.Individual()
ind1.fitness.values = (2.3, 5.6, 7.8)
ind2 = creator.Individual()
ind2.fitness.values = (12.3, 15.6, 1.8)

objs = [0, 1]
res = ind1.fitness.dominates(ind2.fitness)
display(res)


def _dominates(self, other, obj):
    not_equal = False
    for self_wvalue, other_wvalue in zip(np.array(self.wvalues)[obj], np.array(other.wvalues)[obj]):
        if self_wvalue > other_wvalue:
            not_equal = True
        elif self_wvalue < other_wvalue:
            return False
    return not_equal


if getattr(creator.FitnessMin, "dominates", None) is not None:
    setattr(creator.FitnessMin, "dominates", _dominates)

ind1.fitness.dominates(ind2.fitness, objs)


False

True

In [1]:
import carla
import random

client = carla.Client("172.30.32.1", 2000)
client.load_world("Town01")
world = client.get_world()
vehicle_blueprints = world.get_blueprint_library().filter("*vehicle*")
spawn_point = random.choice(world.get_map().get_spawn_points())
ego_vehicle = world.spawn_actor(random.choice(vehicle_blueprints), spawn_point)
pedestrian_blueprints = world.get_blueprint_library().filter("*pedestrian*")
if -10 < spawn_point.rotation.yaw < 10:
    offset = carla.Location(x=7, y=1, z=2)
elif 80 < spawn_point.rotation.yaw < 100:
    offset = carla.Location(x=-1, y=7, z=2)
elif 170 < spawn_point.rotation.yaw < 190:
    offset = carla.Location(x=-7, y=-1, z=2)
else:
    offset = carla.Location(x=1, y=-7, z=2)
pedestrian = world.spawn_actor(
    random.choice(pedestrian_blueprints),
    carla.Transform(spawn_point.location + offset, carla.Rotation(0, spawn_point.rotation.yaw - 90, 0))
)
camera_init_trans = carla.Transform(carla.Location(z=1.5))
camera_bp = world.get_blueprint_library().find("sensor.camera.rgb")
camera_bp.set_attribute("image_size_x", "832")
camera_bp.set_attribute("image_size_y", "640")
camera = world.spawn_actor(camera_bp, camera_init_trans, attach_to=ego_vehicle)
camera.listen(lambda image: image.save_to_disk('out/%06d.png' % image.frame))
camera.id
# camera = world.get_actor(199)
# camera.destroy()

375